# Advanced Problems with Solutions: Context Managers and Lazy Iterators

This notebook develops production-quality patterns for combining **lazy iteration** with **resource management**. The central bug is subtle but common: an iterator may outlive the file, socket, database cursor, or client session that it depends on.

The notebook is fully self-contained. It creates temporary CSV and gzip data, uses only the Python standard library, and includes executable assertions after every solution.

## Learning objectives

By the end, you should be able to:

- explain why returning `csv.reader(file)` from inside `with open(...)` fails;
- design APIs with explicit resource ownership;
- guarantee cleanup after normal exhaustion, early termination, and exceptions;
- build reusable, memory-bounded streaming pipelines;
- use `yield from`, `ExitStack`, `closing`, and `aclosing` correctly;
- distinguish iterators from re-iterable objects;
- test resource safety, not merely output correctness;
- apply the same ideas to compressed files and asynchronous resources.

## Best-practice conventions used here

1. Open text files with an explicit `encoding` and `newline=""` for CSV.
2. Keep resource acquisition and lazy consumption in the same lifetime whenever possible.
3. State who owns a resource: the function or the caller.
4. Close partially consumed generators explicitly when deterministic cleanup matters.
5. Stream by default; materialize only at deliberate boundaries.
6. Validate data at the edge and include row/file context in errors.
7. Test exhaustion, early break, and exception paths.


## Problem map

| # | Topic | Difficulty |
|---:|---|:---:|
| 1 | Diagnose the closed-file iterator bug | ★★☆ |
| 2 | Repair it with a generator | ★★☆ |
| 3 | Eager vs. lazy memory behavior | ★★☆ |
| 4 | Cleanup after early termination | ★★★ |
| 5 | Expose a lazy reader as a context manager | ★★★ |
| 6 | Separate caller-owned and callee-owned resources | ★★★ |
| 7 | Manage many resources with `ExitStack` | ★★★★ |
| 8 | Compose nested generators with `yield from` | ★★★ |
| 9 | Process streams in bounded chunks | ★★★ |
| 10 | Build a lazy pipeline with backpressure | ★★★★ |
| 11 | Understand the hidden cost of `itertools.tee` | ★★★★ |
| 12 | Build a re-iterable CSV collection | ★★★ |
| 13 | Prove cleanup when parsing raises | ★★★★ |
| 14 | Stream compressed CSV safely | ★★★ |
| 15 | Manage asynchronous lazy resources | ★★★★ |
| 16 | Design a bounded look-ahead iterator | ★★★★ |
| 17 | Capstone: resilient multi-file ETL pipeline | ★★★★★ |
| 18 | Resource-safety test suite and design review | ★★★★★ |


## Setup: deterministic temporary data


In [27]:
from __future__ import annotations

import asyncio
import csv
import gc
import gzip
import heapq
import io
import itertools
import json
import tracemalloc
from collections import Counter, OrderedDict, deque
from contextlib import ExitStack, aclosing, contextmanager
from dataclasses import dataclass, field
from pathlib import Path
from tempfile import TemporaryDirectory
from typing import (
    AsyncIterator,
    ContextManager,
    Iterable,
    Iterator,
    Mapping,
    Sequence,
    TextIO,
    TypeVar,
)

T = TypeVar("T")

_tmp = TemporaryDirectory(prefix="lazy_iterators_")
WORKDIR = Path(_tmp.name)


def write_csv(path: Path, header: Sequence[str], rows: Iterable[Sequence[object]]) -> Path:
    """Write a UTF-8 CSV file and return its path."""
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerows(rows)
    return path


TICKETS_HEADER = ["ticket_id", "state", "make", "violation_code", "fine"]
TICKETS_ROWS = [
    (1001, "NY", "HONDA", 21, "65.00"),
    (1002, "NJ", "TOYOTA", 14, "115.00"),
    (1003, "NY", "FORD", 21, "65.00"),
    (1004, "CT", "HONDA", 7, "50.00"),
    (1005, "NY", "TOYOTA", 20, "60.00"),
]

TICKETS_PATH = write_csv(WORKDIR / "tickets.csv", TICKETS_HEADER, TICKETS_ROWS)

# print(f"Working directory: {WORKDIR}")
print(TICKETS_PATH.read_text(encoding="utf-8"))


ticket_id,state,make,violation_code,fine
1001,NY,HONDA,21,65.00
1002,NJ,TOYOTA,14,115.00
1003,NY,FORD,21,65.00
1004,CT,HONDA,7,50.00
1005,NY,TOYOTA,20,60.00



---
## Problem 1 — Diagnose the closed-file iterator bug

The following function looks reasonable, but iteration fails:

```python
def broken_rows(path):
    with path.open() as file:
        return csv.reader(file)
```

**Tasks**

1. Reproduce the failure without stopping the notebook.
2. Explain the exact order of events.
3. Identify the object that owns the operating-system resource.


In [28]:
def broken_rows(path: Path) -> Iterator[list[str]]:
    with path.open("r", encoding="utf-8", newline="") as file:
        return csv.reader(file)


broken = broken_rows(TICKETS_PATH)
try:
    print(next(broken))
except ValueError as exc:
    print(type(exc).__name__ + ":", exc)


ValueError: I/O operation on closed file.


### Solution 1

`csv.reader` is lazy: creating it does not read the CSV. The `with` block exits immediately after `return`, so the file is closed before the caller asks the reader for its first row. The reader depends on the file object, and the file object owns the underlying descriptor.

The important distinction is:

- returning a **fully materialized value** from a `with` block is safe;
- returning a **lazy object that still needs the resource** is unsafe unless the resource lifetime also escapes in a controlled way.


In [29]:
assert TICKETS_PATH.exists()
assert isinstance(broken, Iterator)
print("Diagnosis confirmed: construction succeeded; deferred consumption failed.")


Diagnosis confirmed: construction succeeded; deferred consumption failed.


---
## Problem 2 — Repair the API with a generator

Write `iter_csv_rows(path)` so that:

- it remains lazy;
- it keeps the file open while rows are consumed;
- it closes the file after exhaustion;
- it uses CSV-safe file-opening options.

Then add a second version that optionally skips the header.


In [30]:
def iter_csv_rows(path: Path) -> Iterator[list[str]]:
    """Yield CSV rows while owning the file for the generator's lifetime."""
    with path.open("r", encoding="utf-8", newline="") as file:
        yield from csv.reader(file)


def iter_csv_data_rows(path: Path, *, skip_header: bool = True) -> Iterator[list[str]]:
    rows = iter_csv_rows(path)
    if skip_header:
        next(rows, None)
    yield from rows


all_rows = list(iter_csv_rows(TICKETS_PATH))
data_rows = list(iter_csv_data_rows(TICKETS_PATH))

assert all_rows[0] == TICKETS_HEADER
assert len(data_rows) == len(TICKETS_ROWS)
print(all_rows[:3])


[['ticket_id', 'state', 'make', 'violation_code', 'fine'], ['1001', 'NY', 'HONDA', '21', '65.00'], ['1002', 'NJ', 'TOYOTA', '14', '115.00']]


### Solution 2 discussion

A generator function does not execute when called. It starts on the first `next()`. Therefore the `with` block is entered during consumption, not during generator construction. On normal exhaustion, Python exits the block and closes the file.

Notice the deliberately placed materialization boundary: `list(...)` appears only in the test, not inside the reusable reader.


---
## Problem 3 — Compare eager and lazy memory behavior

Implement:

- `read_all_lines(path)`: returns every line in a list;
- `iter_lines(path)`: yields one line at a time.

Generate a moderately large file and compare peak traced memory while computing the same checksum. Do not assert an exact byte ratio because allocator behavior varies by Python build.


In [31]:
LARGE_PATH = WORKDIR / "large_numbers.txt"
with LARGE_PATH.open("w", encoding="utf-8", newline="") as file:
    for number in range(50_000):
        file.write(f"{number:08d}\n")


def read_all_lines(path: Path) -> list[str]:
    with path.open("r", encoding="utf-8") as file:
        return file.readlines()


def iter_lines(path: Path) -> Iterator[str]:
    with path.open("r", encoding="utf-8") as file:
        for line in file:
            yield line


def measure_peak_bytes(operation) -> tuple[int, int]:
    gc.collect()
    tracemalloc.start()
    checksum = operation()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return checksum, peak


eager_checksum, eager_peak = measure_peak_bytes(
    lambda: sum(int(line) for line in read_all_lines(LARGE_PATH))
)
lazy_checksum, lazy_peak = measure_peak_bytes(
    lambda: sum(int(line) for line in iter_lines(LARGE_PATH))
)

assert eager_checksum == lazy_checksum
print(f"checksum:   {lazy_checksum:,}")
print(f"eager peak: {eager_peak:,} bytes")
print(f"lazy peak:  {lazy_peak:,} bytes")
print(f"ratio:      {eager_peak / max(lazy_peak, 1):.1f}x")


checksum:   1,249,975,000
eager peak: 2,962,813 bytes
lazy peak:  26,929 bytes
ratio:      110.0x


### Solution 3 discussion

Both implementations are correct, but they have different contracts. The eager version transfers all data into memory and can safely return after closing the file. The lazy version retains the resource during iteration and usually uses memory proportional to one line plus pipeline state.

Choose eager loading when the dataset is known to be small and repeated random access matters. Choose streaming when input size may be large, only a prefix may be needed, or latency to the first result matters.


---
## Problem 4 — Guarantee cleanup after early termination

Normal exhaustion is easy. The harder case is a consumer that reads one item and stops.

Create a generator that records open/close events. Demonstrate:

1. the resource is open after the first item;
2. `generator.close()` runs the generator's `finally` block;
3. `contextlib.closing` gives deterministic cleanup around partial consumption.


In [32]:
def iter_lines_with_events(path: Path, events: list[tuple[str, bool]]) -> Iterator[str]:
    file = path.open("r", encoding="utf-8")
    events.append(("opened", file.closed))
    try:
        for line in file:
            yield line.rstrip("\n")
    finally:
        file.close()
        events.append(("closed", file.closed))


events: list[tuple[str, bool]] = []
generator = iter_lines_with_events(LARGE_PATH, events)
first = next(generator)
assert first == "00000000"
assert events == [("opened", False)]

generator.close()
assert events[-1] == ("closed", True)
print(events)


[('opened', False), ('closed', True)]


In [33]:
from contextlib import closing

closing_events: list[tuple[str, bool]] = []
with closing(iter_lines_with_events(LARGE_PATH, closing_events)) as lines:
    preview = list(itertools.islice(lines, 3))

assert preview == ["00000000", "00000001", "00000002"]
assert closing_events[-1] == ("closed", True)
print(preview, closing_events)


['00000000', '00000001', '00000002'] [('opened', False), ('closed', True)]


### Solution 4 discussion

A generator's `finally` block runs when it is exhausted, explicitly closed, or finalized. Relying on finalization timing is weaker than explicit closure, especially outside CPython. Use `closing(...)` when a generator owns an important resource and the consumer may stop early.

A plain `for` loop that uses `break` does not automatically call `close()` on an arbitrary generator object that remains referenced.


---
## Problem 5 — Expose a lazy reader as a context manager

Sometimes callers need the raw `csv.reader` API rather than a wrapper generator. Implement `open_csv_reader(path)` as a context manager so the valid lifetime is explicit at the call site.


In [34]:
@contextmanager
def open_csv_reader(path: Path) -> Iterator[csv._reader]:
    """Yield a CSV reader that is valid only inside the with-block."""
    with path.open("r", encoding="utf-8", newline="") as file:
        yield csv.reader(file)


with open_csv_reader(TICKETS_PATH) as reader:
    header = next(reader)
    first_two = list(itertools.islice(reader, 2))

assert header == TICKETS_HEADER
assert first_two[0][0] == "1001"
print(header)
print(first_two)


['ticket_id', 'state', 'make', 'violation_code', 'fine']
[['1001', 'NY', 'HONDA', '21', '65.00'], ['1002', 'NJ', 'TOYOTA', '14', '115.00']]


### Solution 5 discussion

This API makes misuse harder because the resource boundary is visible:

```python
with open_csv_reader(path) as reader:
    ...
```

Do not return `reader` from inside this outer `with`; that would recreate the original lifetime bug one level higher.


---
## Problem 6 — Separate caller-owned and callee-owned resources

Design two APIs:

- `iter_csv_stream(stream)`: the caller owns the already-open stream; the function must not close it;
- `iter_csv_path(path)`: the function owns the file and must close it.

This split makes ownership testable and avoids functions that sometimes close an argument and sometimes do not.


In [35]:
def iter_csv_stream(stream: TextIO) -> Iterator[list[str]]:
    """Read from a caller-owned text stream without closing it."""
    yield from csv.reader(stream)


def iter_csv_path(path: Path) -> Iterator[list[str]]:
    """Read from a function-owned path and close the opened file."""
    with path.open("r", encoding="utf-8", newline="") as stream:
        yield from iter_csv_stream(stream)


caller_stream = io.StringIO("a,b\n1,2\n")
rows = list(iter_csv_stream(caller_stream))
assert rows == [["a", "b"], ["1", "2"]]
assert not caller_stream.closed
caller_stream.close()

assert list(iter_csv_path(TICKETS_PATH))[0] == TICKETS_HEADER
print("Caller-owned stream remained open; path-owned stream was managed internally.")


Caller-owned stream remained open; path-owned stream was managed internally.


### Solution 6 discussion

Resource ownership is part of an API's semantics. A useful naming convention is:

- `..._stream(file)` borrows a resource;
- `..._path(path)` acquires and owns a resource.

This also improves testing because `io.StringIO` can exercise parsing logic without touching the filesystem.


---
## Problem 7 — Manage many resources with `ExitStack`

Three files contain sorted integers. Write a lazy merge that:

- opens all files under one managed lifetime;
- converts each line to `int` lazily;
- yields globally sorted values using `heapq.merge`;
- closes everything on exhaustion or explicit generator closure.


In [36]:
NUMBER_PATHS: list[Path] = []
for index, values in enumerate(([1, 4, 9], [2, 5, 6, 8], [0, 3, 7, 10])):
    path = WORKDIR / f"sorted_{index}.txt"
    path.write_text("".join(f"{value}\n" for value in values), encoding="utf-8")
    NUMBER_PATHS.append(path)


def merge_sorted_files(paths: Sequence[Path]) -> Iterator[int]:
    with ExitStack() as stack:
        streams = [
            stack.enter_context(path.open("r", encoding="utf-8"))
            for path in paths
        ]
        # map binds each stream immediately; nested generator expressions here
        # would otherwise risk late-binding the loop variable.
        integer_streams = [map(int, stream) for stream in streams]
        yield from heapq.merge(*integer_streams)


merged = list(merge_sorted_files(NUMBER_PATHS))
assert merged == list(range(11))
print(merged)


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


### Solution 7 discussion

`ExitStack` is the scalable form of nested `with` statements. It is especially useful when the number of resources is dynamic. Because the `ExitStack` lives inside the generator, all entered contexts remain active during iteration.

For thousands of files, opening everything at once may exceed descriptor limits. A production design would perform a multi-stage merge with a bounded fan-in.


---
## Problem 8 — Compose nested generators with `yield from`

Create three CSV files and implement a directory reader that opens **only one file at a time**. Instrument it to prove the maximum number of simultaneously open files is one.


In [37]:
PARTS_DIR = WORKDIR / "parts"
PARTS_DIR.mkdir()
for part in range(3):
    write_csv(
        PARTS_DIR / f"part_{part}.csv",
        ["part", "value"],
        [(part, part * 10 + offset) for offset in range(3)],
    )


@dataclass
class OpenCounter:
    current: int = 0
    maximum: int = 0

    @contextmanager
    def open_text(self, path: Path) -> Iterator[TextIO]:
        file = path.open("r", encoding="utf-8", newline="")
        self.current += 1
        self.maximum = max(self.maximum, self.current)
        try:
            yield file
        finally:
            file.close()
            self.current -= 1


def iter_part_rows(path: Path, counter: OpenCounter) -> Iterator[dict[str, str]]:
    with counter.open_text(path) as file:
        yield from csv.DictReader(file)


def iter_directory_rows(directory: Path, counter: OpenCounter) -> Iterator[dict[str, str]]:
    for path in sorted(directory.glob("*.csv")):
        yield from iter_part_rows(path, counter)


counter = OpenCounter()
part_rows = list(iter_directory_rows(PARTS_DIR, counter))
assert len(part_rows) == 9
assert counter.current == 0
assert counter.maximum == 1
print(part_rows[:4])
print("maximum simultaneously open files:", counter.maximum)


[{'part': '0', 'value': '0'}, {'part': '0', 'value': '1'}, {'part': '0', 'value': '2'}, {'part': '1', 'value': '10'}]
maximum simultaneously open files: 1


### Solution 8 discussion

`yield from` delegates iteration while preserving generator control flow. Each inner generator is exhausted before the next file is opened, so the design is descriptor-efficient. It also propagates `close()` into the active subgenerator.


---
## Problem 9 — Process a stream in bounded chunks

Implement a generic `batched(iterable, size)` helper without loading the entire input. Then compute sums of 1,000-number chunks from the large text file.

Requirements:

- reject non-positive sizes;
- return tuples;
- emit the final partial batch;
- consume input only as requested.


In [38]:
def batched(iterable: Iterable[T], size: int) -> Iterator[tuple[T, ...]]:
    if size <= 0:
        raise ValueError("size must be positive")
    iterator = iter(iterable)
    while batch := tuple(itertools.islice(iterator, size)):
        yield batch


number_stream = (int(line) for line in iter_lines(LARGE_PATH))
chunk_sums = [sum(batch) for batch in batched(number_stream, 1_000)]

assert len(chunk_sums) == 50
assert chunk_sums[0] == sum(range(1_000))
assert chunk_sums[-1] == sum(range(49_000, 50_000))
print(chunk_sums[:3], "...", chunk_sums[-3:])


[499500, 1499500, 2499500] ... [47499500, 48499500, 49499500]


### Solution 9 discussion

Chunking introduces a deliberate, bounded materialization point. Memory is `O(size)`, not `O(total input)`. This pattern is useful for batched database writes, vectorized transforms, and rate-limited APIs.


---
## Problem 10 — Build a lazy pipeline with backpressure

Construct a pipeline that:

1. reads integers lazily;
2. keeps multiples of 7;
3. squares them;
4. computes a rolling mean over the latest 4 values;
5. returns only the first 5 means.

Instrument the source to show that the pipeline does not read all 50,000 lines.


In [39]:
def counted_integers(path: Path, metrics: dict[str, int]) -> Iterator[int]:
    """Yield integers and explicitly propagate close() to the file-owning iterator."""
    lines = iter_lines(path)
    try:
        for line in lines:
            metrics["lines_read"] += 1
            yield int(line)
    finally:
        # Essential when a downstream consumer stops early.
        lines.close()


def rolling_mean(values: Iterable[float], window: int) -> Iterator[float]:
    if window <= 0:
        raise ValueError("window must be positive")
    recent: deque[float] = deque(maxlen=window)
    total = 0.0
    for value in values:
        if len(recent) == window:
            total -= recent[0]
        recent.append(value)
        total += value
        if len(recent) == window:
            yield total / window


metrics = {"lines_read": 0}

# Every stage is a generator with close(). ExitStack closes the pipeline in
# reverse order even though islice intentionally consumes only a small prefix.
with ExitStack() as stack:
    source = stack.enter_context(
        closing(counted_integers(LARGE_PATH, metrics))
    )
    multiples_of_seven = stack.enter_context(
        closing((value for value in source if value % 7 == 0))
    )
    squares = stack.enter_context(
        closing((value * value for value in multiples_of_seven))
    )
    means = stack.enter_context(
        closing(rolling_mean(squares, window=4))
    )

    first_five_means = list(itertools.islice(means, 5))

assert len(first_five_means) == 5
assert metrics["lines_read"] < 100
print("means:", first_five_means)
print("source lines read:", metrics["lines_read"], "of 50,000")
print("All partially consumed pipeline stages were closed explicitly.")


means: [171.5, 367.5, 661.5, 1053.5, 1543.5]
source lines read: 50 of 50,000
All partially consumed pipeline stages were closed explicitly.


### Solution 10 discussion

Backpressure means downstream demand controls upstream work. `islice(..., 5)` requests only enough rolling means to satisfy five outputs. Every upstream stage processes only what is necessary.

Early termination creates a second responsibility: **close the partially consumed pipeline**. Here, `ExitStack` registers each generator with `closing(...)` and closes the stages in reverse order. The root generator also propagates `close()` to `iter_lines`, which releases the file immediately.

This is particularly important on Windows, where an open file handle can cause `TemporaryDirectory.cleanup()` to fail with `PermissionError: [WinError 32]`.

A single accidental `list(...)` in the middle would destroy backpressure, while omitting explicit closure could leak the resource after a prefix-only read.


---
## Problem 11 — Understand the hidden cost of `itertools.tee`

`tee` creates logically independent iterators, but it may buffer items when consumers advance at different speeds.

Create a counted source, split it, consume one branch far ahead, and observe that the lagging branch can later read old values without advancing the original source.


In [40]:
def counted_range(stop: int, metrics: dict[str, int]) -> Iterator[int]:
    for value in range(stop):
        metrics["produced"] += 1
        yield value


tee_metrics = {"produced": 0}
left, right = itertools.tee(counted_range(20, tee_metrics), 2)

left_prefix = list(itertools.islice(left, 12))
produced_after_left = tee_metrics["produced"]
right_prefix = list(itertools.islice(right, 3))
produced_after_right = tee_metrics["produced"]

assert left_prefix == list(range(12))
assert right_prefix == [0, 1, 2]
assert produced_after_left == 12
assert produced_after_right == 12
print({
    "left_consumed": len(left_prefix),
    "right_consumed": len(right_prefix),
    "source_produced": tee_metrics["produced"],
    "minimum_items_buffered_for_right": produced_after_left - len(right_prefix),
})


{'left_consumed': 12, 'right_consumed': 3, 'source_produced': 12, 'minimum_items_buffered_for_right': 9}


### Solution 11 discussion

The source produced 12 values for the leading branch. The lagging branch then read values from `tee`'s internal buffer, so the source count did not increase. If the lead becomes very large or unbounded, memory use can also become very large or unbounded.

Prefer a one-pass combined aggregation when possible. Use `tee` only when the distance between consumers is known to remain small or the buffering cost is acceptable.


In [41]:
def sum_and_count(values: Iterable[int]) -> tuple[int, int]:
    """One-pass alternative to duplicating a stream for two aggregations."""
    total = 0
    count = 0
    for value in values:
        total += value
        count += 1
    return total, count

assert sum_and_count(range(10)) == (45, 10)


---
## Problem 12 — Build a re-iterable CSV collection

A generator object is one-shot. Build `CSVRows`, a lightweight re-iterable object whose `__iter__` opens a fresh file each time. It should support repeated scans without storing all rows.


In [42]:
@dataclass(frozen=True)
class CSVRows(Iterable[dict[str, str]]):
    path: Path

    def __iter__(self) -> Iterator[dict[str, str]]:
        with self.path.open("r", encoding="utf-8", newline="") as file:
            yield from csv.DictReader(file)


collection = CSVRows(TICKETS_PATH)
ny_count = sum(row["state"] == "NY" for row in collection)
honda_count = sum(row["make"] == "HONDA" for row in collection)

assert ny_count == 3
assert honda_count == 2
print({"NY": ny_count, "HONDA": honda_count})


{'NY': 3, 'HONDA': 2}


### Solution 12 discussion

`CSVRows` is an **iterable**, not an iterator. Every call to `iter(collection)` creates a new generator and opens a new file. This provides re-iterability without caching the dataset.

Trade-off: repeated scans repeat I/O. If repeated access is frequent and the data is small, an eager immutable collection may be a better abstraction.


---
## Problem 13 — Prove cleanup when parsing raises

Create a CSV containing an invalid integer. Implement a typed parser that raises a contextual error and prove the file closes even though iteration terminates with an exception.


In [43]:
BAD_PATH = write_csv(
    WORKDIR / "bad_tickets.csv",
    TICKETS_HEADER,
    [
        (2001, "NY", "FORD", 21, "65.00"),
        (2002, "NJ", "HONDA", "NOT_AN_INT", "50.00"),
        (2003, "NY", "TOYOTA", 14, "115.00"),
    ],
)


class RowParseError(ValueError):
    pass


@dataclass(frozen=True)
class Ticket:
    ticket_id: int
    state: str
    make: str
    violation_code: int
    fine_cents: int


def parse_money_to_cents(text: str) -> int:
    whole, dot, fraction = text.partition(".")
    if not dot:
        fraction = "00"
    fraction = (fraction + "00")[:2]
    return int(whole) * 100 + int(fraction)


def iter_tickets(path: Path, events: list[str] | None = None) -> Iterator[Ticket]:
    events = events if events is not None else []
    file = path.open("r", encoding="utf-8", newline="")
    events.append("opened")
    try:
        for line_number, row in enumerate(csv.DictReader(file), start=2):
            try:
                yield Ticket(
                    ticket_id=int(row["ticket_id"]),
                    state=row["state"].strip().upper(),
                    make=row["make"].strip().upper(),
                    violation_code=int(row["violation_code"]),
                    fine_cents=parse_money_to_cents(row["fine"]),
                )
            except (KeyError, TypeError, ValueError) as exc:
                raise RowParseError(
                    f"{path.name}: row {line_number}: {exc}; data={row!r}"
                ) from exc
    finally:
        file.close()
        events.append(f"closed={file.closed}")


parse_events: list[str] = []
try:
    list(iter_tickets(BAD_PATH, parse_events))
except RowParseError as exc:
    print(exc)

assert parse_events == ["opened", "closed=True"]
print(parse_events)


bad_tickets.csv: row 3: invalid literal for int() with base 10: 'NOT_AN_INT'; data={'ticket_id': '2002', 'state': 'NJ', 'make': 'HONDA', 'violation_code': 'NOT_AN_INT', 'fine': '50.00'}
['opened', 'closed=True']


### Solution 13 discussion

The `finally` block protects cleanup across normal completion, explicit closure, and exceptions. The raised error preserves the original exception as `__cause__` and adds file name, row number, and offending data.

Avoid silently skipping malformed rows unless the API exposes a deliberate error policy and metrics.


---
## Problem 14 — Stream compressed CSV safely

Write `iter_gzip_csv(path)` for a `.csv.gz` file. Use text mode, UTF-8, and `newline=""`. Verify that the reader is lazy and produces the original rows.


In [44]:
GZIP_PATH = WORKDIR / "tickets.csv.gz"
with gzip.open(GZIP_PATH, "wt", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(TICKETS_HEADER)
    writer.writerows(TICKETS_ROWS)


def iter_gzip_csv(path: Path) -> Iterator[dict[str, str]]:
    with gzip.open(path, "rt", encoding="utf-8", newline="") as file:
        yield from csv.DictReader(file)


gzip_rows = list(iter_gzip_csv(GZIP_PATH))
assert len(gzip_rows) == len(TICKETS_ROWS)
assert gzip_rows[0]["ticket_id"] == "1001"
print(gzip_rows[:2])


[{'ticket_id': '1001', 'state': 'NY', 'make': 'HONDA', 'violation_code': '21', 'fine': '65.00'}, {'ticket_id': '1002', 'state': 'NJ', 'make': 'TOYOTA', 'violation_code': '14', 'fine': '115.00'}]


### Solution 14 discussion

Layered resources still need one coherent lifetime: the gzip decompressor depends on the compressed file, and the CSV reader depends on the text stream produced by the decompressor. Keeping all layers inside the generator's `with` block preserves that lifetime.


---
## Problem 15 — Manage asynchronous lazy resources

An async generator may depend on an async client session. Implement a fake client and `iter_pages`. Consume only one page, then use `aclosing` to prove deterministic cleanup.


In [45]:
@dataclass
class FakeAsyncClient:
    events: list[str]

    async def __aenter__(self) -> "FakeAsyncClient":
        self.events.append("client-open")
        return self

    async def __aexit__(self, exc_type, exc, tb) -> None:
        self.events.append("client-close")

    async def fetch(self, page_number: int) -> list[int]:
        await asyncio.sleep(0)
        self.events.append(f"fetch-{page_number}")
        return [page_number * 10 + offset for offset in range(3)]


async def iter_pages(page_count: int, events: list[str]) -> AsyncIterator[list[int]]:
    async with FakeAsyncClient(events) as client:
        for page_number in range(page_count):
            yield await client.fetch(page_number)


async def async_demo() -> tuple[list[int], list[str]]:
    events: list[str] = []
    async with aclosing(iter_pages(5, events)) as pages:
        first_page = await anext(pages)
    return first_page, events


first_page, async_events = await async_demo()
assert first_page == [0, 1, 2]
assert async_events == ["client-open", "fetch-0", "client-close"]
print(first_page, async_events)


[0, 1, 2] ['client-open', 'fetch-0', 'client-close']


### Solution 15 discussion

`aclosing` is the asynchronous counterpart of `closing`. It awaits the async generator's `aclose()` method, allowing the active `async with` block to exit in the same task context.

The resource-lifetime principle is unchanged; only the protocol changes from `__enter__/__exit__` to `__aenter__/__aexit__`.


---
## Problem 16 — Design a bounded look-ahead iterator

Implement `Peekable`, which supports:

- `peek()` without consuming the next item;
- normal iteration;
- at most one buffered element;
- a caller-supplied sentinel default for exhaustion.

This exercise tests the difference between looking ahead and duplicating an entire stream.


In [46]:
_MISSING = object()


class Peekable(Iterator[T]):
    def __init__(self, iterable: Iterable[T]):
        self._iterator = iter(iterable)
        self._buffer: object = _MISSING

    def __iter__(self) -> "Peekable[T]":
        return self

    def __next__(self) -> T:
        if self._buffer is not _MISSING:
            value = self._buffer
            self._buffer = _MISSING
            return value  # type: ignore[return-value]
        return next(self._iterator)

    def peek(self, default: object = _MISSING) -> T:
        if self._buffer is _MISSING:
            try:
                self._buffer = next(self._iterator)
            except StopIteration:
                if default is _MISSING:
                    raise
                return default  # type: ignore[return-value]
        return self._buffer  # type: ignore[return-value]


peekable = Peekable(iter([10, 20, 30]))
assert peekable.peek() == 10
assert peekable.peek() == 10
assert next(peekable) == 10
assert peekable.peek() == 20
assert list(peekable) == [20, 30]
assert peekable.peek(None) is None
print("Peekable passed all checks with a one-item buffer.")


Peekable passed all checks with a one-item buffer.


### Solution 16 discussion

A one-item look-ahead buffer has bounded memory. In contrast, `tee` can buffer the entire distance between consumers. `Peekable` is useful for parsers, grouping logic, and state machines that need limited look-ahead.


---
## Problem 17 — Capstone: resilient multi-file ETL pipeline

Build a single-pass pipeline for ticket files with these requirements:

1. Open at most one input file at a time.
2. Normalize state and vehicle make to uppercase.
3. Validate integer and money fields.
4. Deduplicate ticket IDs using a bounded least-recently-used window.
5. Support error policies: `"raise"`, `"skip"`, and `"collect"`.
6. Aggregate counts and fine totals by state.
7. Record rows read, rows accepted, duplicates, and errors.
8. Keep parsing lazy; aggregate only at the final reduction boundary.

A bounded deduplication window is intentionally not equivalent to global deduplication. It trades perfect recall for bounded memory.


In [47]:
CAPSTONE_DIR = WORKDIR / "capstone"
CAPSTONE_DIR.mkdir()

CAPSTONE_PATHS = [
    write_csv(
        CAPSTONE_DIR / "tickets_a.csv",
        TICKETS_HEADER,
        [
            (3001, "ny", "honda", 21, "65.00"),
            (3002, "NJ", "toyota", 14, "115.00"),
            (3003, "NY", "ford", 21, "65.00"),
        ],
    ),
    write_csv(
        CAPSTONE_DIR / "tickets_b.csv",
        TICKETS_HEADER,
        [
            (3002, "NJ", "TOYOTA", 14, "115.00"),  # recent duplicate
            (3004, "ct", "honda", 7, "50.00"),
            (3005, "NY", "tesla", "BAD", "75.00"),  # invalid code
            (3006, "NY", "tesla", 20, "60.00"),
        ],
    ),
]


In [48]:
@dataclass(frozen=True)
class SourceRow:
    path: Path
    line_number: int
    data: Mapping[str, str]


@dataclass
class PipelineStats:
    rows_read: int = 0
    rows_accepted: int = 0
    duplicates: int = 0
    errors: int = 0
    files_open_now: int = 0
    max_files_open: int = 0
    collected_errors: list[str] = field(default_factory=list)


@dataclass(frozen=True)
class ETLSummary:
    stats: PipelineStats
    count_by_state: dict[str, int]
    fine_cents_by_state: dict[str, int]


def iter_source_rows(paths: Sequence[Path], stats: PipelineStats) -> Iterator[SourceRow]:
    for path in paths:
        file = path.open("r", encoding="utf-8", newline="")
        stats.files_open_now += 1
        stats.max_files_open = max(stats.max_files_open, stats.files_open_now)
        try:
            for line_number, row in enumerate(csv.DictReader(file), start=2):
                stats.rows_read += 1
                yield SourceRow(path=path, line_number=line_number, data=row)
        finally:
            file.close()
            stats.files_open_now -= 1


def parse_source_row(source: SourceRow) -> Ticket:
    row = source.data
    try:
        return Ticket(
            ticket_id=int(row["ticket_id"]),
            state=row["state"].strip().upper(),
            make=row["make"].strip().upper(),
            violation_code=int(row["violation_code"]),
            fine_cents=parse_money_to_cents(row["fine"]),
        )
    except (KeyError, TypeError, ValueError) as exc:
        raise RowParseError(
            f"{source.path.name}: row {source.line_number}: {exc}; data={dict(row)!r}"
        ) from exc


def iter_valid_tickets(
    rows: Iterable[SourceRow],
    stats: PipelineStats,
    *,
    error_policy: str,
) -> Iterator[Ticket]:
    if error_policy not in {"raise", "skip", "collect"}:
        raise ValueError("error_policy must be 'raise', 'skip', or 'collect'")

    for source in rows:
        try:
            yield parse_source_row(source)
        except RowParseError as exc:
            stats.errors += 1
            if error_policy == "raise":
                raise
            if error_policy == "collect":
                stats.collected_errors.append(str(exc))


def iter_recently_unique(
    tickets: Iterable[Ticket],
    stats: PipelineStats,
    *,
    window_size: int,
) -> Iterator[Ticket]:
    if window_size <= 0:
        raise ValueError("window_size must be positive")

    recent: OrderedDict[int, None] = OrderedDict()
    for ticket in tickets:
        if ticket.ticket_id in recent:
            stats.duplicates += 1
            recent.move_to_end(ticket.ticket_id)
            continue

        recent[ticket.ticket_id] = None
        if len(recent) > window_size:
            recent.popitem(last=False)
        yield ticket


def run_ticket_etl(
    paths: Sequence[Path],
    *,
    error_policy: str = "collect",
    dedupe_window: int = 10_000,
) -> ETLSummary:
    stats = PipelineStats()
    source_rows = iter_source_rows(paths, stats)
    valid = iter_valid_tickets(source_rows, stats, error_policy=error_policy)
    unique = iter_recently_unique(valid, stats, window_size=dedupe_window)

    count_by_state: Counter[str] = Counter()
    fine_cents_by_state: Counter[str] = Counter()

    for ticket in unique:
        stats.rows_accepted += 1
        count_by_state[ticket.state] += 1
        fine_cents_by_state[ticket.state] += ticket.fine_cents

    return ETLSummary(
        stats=stats,
        count_by_state=dict(count_by_state),
        fine_cents_by_state=dict(fine_cents_by_state),
    )


In [49]:
summary = run_ticket_etl(
    CAPSTONE_PATHS,
    error_policy="collect",
    dedupe_window=100,
)

assert summary.stats.rows_read == 7
assert summary.stats.rows_accepted == 5
assert summary.stats.duplicates == 1
assert summary.stats.errors == 1
assert summary.stats.files_open_now == 0
assert summary.stats.max_files_open == 1
assert summary.count_by_state == {"NY": 3, "NJ": 1, "CT": 1}
assert summary.fine_cents_by_state == {
    "NY": 19_000,
    "NJ": 11_500,
    "CT": 5_000,
}

print(json.dumps({
    "stats": {
        "rows_read": summary.stats.rows_read,
        "rows_accepted": summary.stats.rows_accepted,
        "duplicates": summary.stats.duplicates,
        "errors": summary.stats.errors,
        "max_files_open": summary.stats.max_files_open,
    },
    "count_by_state": summary.count_by_state,
    "fine_cents_by_state": summary.fine_cents_by_state,
    "collected_errors": summary.stats.collected_errors,
}, indent=2))


{
  "stats": {
    "rows_read": 7,
    "rows_accepted": 5,
    "duplicates": 1,
    "errors": 1,
    "max_files_open": 1
  },
  "count_by_state": {
    "NY": 3,
    "NJ": 1,
    "CT": 1
  },
  "fine_cents_by_state": {
    "NY": 19000,
    "NJ": 11500,
    "CT": 5000
  },
  "collected_errors": [
    "tickets_b.csv: row 4: invalid literal for int() with base 10: 'BAD'; data={'ticket_id': '3005', 'state': 'NY', 'make': 'tesla', 'violation_code': 'BAD', 'fine': '75.00'}"
  ]
}


### Solution 17 discussion

The capstone separates concerns into lazy stages:

- acquisition and source context;
- parsing and normalization;
- error policy;
- bounded deduplication;
- final reduction.

Each stage is independently testable. The final counters are eager because aggregation is a deliberate terminal operation. The pipeline opens one file at a time and closes the active file even if a downstream stage raises.

For global deduplication, use durable external state such as a database uniqueness constraint, partitioned disk-backed set, or sort-and-merge process. An in-memory set is exact but grows with every unique ID.


---
## Problem 18 — Resource-safety test suite and design review

Output correctness alone is insufficient. Add tests for:

1. normal exhaustion;
2. early explicit closure;
3. parse-time exception;
4. repeatability of an iterable;
5. bounded file concurrency;
6. invalid configuration.

Then review several tempting designs and identify the failure mode.


In [50]:
def test_normal_exhaustion() -> None:
    events: list[tuple[str, bool]] = []
    assert len(list(iter_lines_with_events(TICKETS_PATH, events))) > 0
    assert events[-1] == ("closed", True)


def test_early_close() -> None:
    events: list[tuple[str, bool]] = []
    generator = iter_lines_with_events(TICKETS_PATH, events)
    next(generator)
    generator.close()
    assert events[-1] == ("closed", True)


def test_exception_cleanup() -> None:
    events: list[str] = []
    try:
        list(iter_tickets(BAD_PATH, events))
    except RowParseError:
        pass
    else:
        raise AssertionError("RowParseError was expected")
    assert events[-1] == "closed=True"


def test_reiterable_collection() -> None:
    rows = CSVRows(TICKETS_PATH)
    assert sum(1 for _ in rows) == sum(1 for _ in rows) == 5


def test_bounded_concurrency() -> None:
    stats = run_ticket_etl(CAPSTONE_PATHS).stats
    assert stats.files_open_now == 0
    assert stats.max_files_open == 1


def test_invalid_configuration() -> None:
    try:
        list(batched([1, 2, 3], 0))
    except ValueError:
        pass
    else:
        raise AssertionError("ValueError was expected")

    try:
        run_ticket_etl(CAPSTONE_PATHS, error_policy="ignore")
    except ValueError:
        pass
    else:
        raise AssertionError("ValueError was expected")


TESTS = [
    test_normal_exhaustion,
    test_early_close,
    test_exception_cleanup,
    test_reiterable_collection,
    test_bounded_concurrency,
    test_invalid_configuration,
]

for test in TESTS:
    test()
    print("PASS", test.__name__)


PASS test_normal_exhaustion
PASS test_early_close
PASS test_exception_cleanup
PASS test_reiterable_collection
PASS test_bounded_concurrency
PASS test_invalid_configuration


### Design review answers

#### A. `return map(parse, file)` inside `with open(...)`

**Bug:** `map` is lazy and still depends on the closed file. Use a generator with the `with` inside it, or materialize before leaving the block.

#### B. Accept either a path or open stream, and always call `.close()`

**Bug:** ownership is ambiguous; the function may close a caller-owned stream. Prefer separate APIs or an explicit `close_stream: bool = False` contract.

#### C. Use `list(reader)` to eliminate all lifetime issues

**Trade-off, not always a bug:** correct for small inputs, dangerous for unbounded or large inputs, and increases latency to the first result.

#### D. Depend on garbage collection to close a partially consumed generator

**Bug-prone:** cleanup timing is implementation-dependent. Use `close()`, `closing(...)`, or an explicit context-manager API.

#### E. Use `tee` to let ten consumers process a huge live stream at unrelated speeds

**Risk:** internal buffering can grow without bound. Use a broker/queue with backpressure, durable storage, or coordinated consumers.

#### F. Open every file in a directory and merge them

**Risk:** descriptor exhaustion. Use bounded fan-in or process one file at a time when global simultaneous access is unnecessary.


---
# Additional challenge problems

These are intentionally left as extension exercises, but each includes a target behavior so you can test your implementation.

1. **Checkpointable reader:** yield `(byte_offset, decoded_record)` and resume from a saved offset. Account for text encoding and CSV records that may span physical lines.
2. **Transactional batch writer:** consume input in chunks, write each chunk to a temporary file, and atomically replace the destination only after successful completion.
3. **Timeout-aware async stream:** add an `asyncio.timeout(...)` boundary while guaranteeing `aclose()` and client cleanup.
4. **Bounded fan-in merge:** merge 10,000 sorted files while never opening more than 32 at once.
5. **Quarantine policy:** send malformed records to a separate CSV with original file name, line number, raw data, and error message.
6. **Schema evolution:** accept two header versions and normalize both into one dataclass without loading entire files.
7. **Cancellation test:** cancel a task during async iteration and prove the async resource still closes.
8. **Observable pipeline:** add per-stage counters and elapsed-time measurements without forcing materialization.
9. **Resource-owning iterator class:** implement `__enter__`, `__exit__`, `__iter__`, `__next__`, and idempotent `close()`.
10. **External-memory deduplication:** replace the bounded LRU with SQLite and a unique index while preserving streaming input.


# Final checklist

Before returning or storing a lazy iterator, ask:

- What resource does this iterator still need?
- Who owns that resource?
- When exactly is it released?
- What happens on early break?
- What happens if transformation code raises?
- Is the object one-shot or re-iterable?
- Is buffering bounded?
- Are file encoding and CSV newline rules explicit?
- Do tests cover cleanup, not only values?

The central rule is simple: **a lazy computation and every resource it depends on must share a valid lifetime.**


## Cleanup


In [51]:
# Safe after all examples, including prefix-only lazy pipelines.
# Explicit generator cleanup in Problem 10 ensures Windows has no locked files.
gc.collect()
_tmp.cleanup()
print("Temporary resources removed.")


Temporary resources removed.
